# QueryEngine and Visualization Integration

## Install

```bash
pip install graphyco
```

---

## Setup & Benchmark Loading

Initialize QueryEngine on canonical benchmark dataset.

In [1]:
import torch
import torch.nn as nn
import pandas as pd
from graphyco import extract_benchmark_json, QueryEngine


# Define canonical architectures inline
class ResNetBlock(nn.Module):
    def __init__(self, dim=64):
        super().__init__()
        self.l1 = nn.Linear(dim, dim)
        self.r1 = nn.ReLU()
        self.l2 = nn.Linear(dim, dim)
        self.r2 = nn.ReLU()
        self.out = nn.Linear(dim, 10)
    def forward(self, x):
        h = self.l2(self.r1(self.l1(x)))
        return self.out(self.r2(h + x))


models = {
    "Sequential MLP": (nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 10)), torch.randn(8, 64)),
    "ResNet Block": (ResNetBlock(dim=64), torch.randn(8, 64)),
}

# Generate benchmark data for both architectures
benchmark = {"architectures": {}, "step_trajectories": {}, "ablation_validation": {}}
for name, (model, inp) in models.items():
    sub = extract_benchmark_json(model, inputs=inp, steps=5, arch_name=name)
    benchmark["architectures"].update(sub["architectures"])
    benchmark["step_trajectories"].update(sub["step_trajectories"])
    benchmark["ablation_validation"].update(sub["ablation_validation"])

engine = QueryEngine(benchmark)
archs = engine.list_architectures()
print(f"Loaded {len(archs)} Architectures: {archs}")


Loaded 2 Architectures: ['Sequential MLP', 'ResNet Block']


## Architecture Invariant Summary (`describe`)

Inspect formatted DataFrame summary of topological invariants.

In [2]:
engine.set_default("ResNet Block")
df_summary = engine.describe()
df_summary

,Value,Description
Metric,,
Graph Density (D),0.1428,Edge connectivity ratio |E| / (|V|(|V|-1))
Flow Coherence (C),0.9956,Conservation of incoming vs outgoing gradient ...
Topological Bottleneck,1.5000,Peak node betweenness centrality in DAG
Structural Resilience (R),0.3333,Path connectivity retention under single-node ...
Dynamic Bottleneck (G_max),1.0000,Peak gradient concentration factor relative to...
Bottleneck Node,out,Layer exhibiting maximal gradient concentration
Spearman Rank (rho_AG),0.0485,Rank correlation between forward activations a...
Mean Stability (CV),0.0085,Coefficient of variation across recorded steps


## Cross-Architecture Sibling Comparison

Compare invariants across all canonical models side-by-side.

In [3]:
df_comparison = engine.describe(siblings=True)
df_comparison

,Sequential MLP,ResNet Block
Metric,,
Graph Density (D),0.1428,0.1428
Flow Coherence (C),0.9934,0.9956
Topological Bottleneck,1.167,1.5
Structural Resilience (R),0.3684,0.3333
Dynamic Bottleneck (G_max),1.0,1.0
Bottleneck Node,_4,out
Spearman Rank (rho_AG),-1.0,0.0485
Mean Stability (CV),0.0064,0.0085


## Comparative Bottleneck Queries

Extract topological and dynamic gradient bottlenecks across models.

In [4]:
bottlenecks = engine.get_bottlenecks()
pd.DataFrame(bottlenecks)

,architecture,topological_bottleneck,gradient_bottleneck_gmax,bottleneck_node,structural_resilience,coherence
0,Sequential MLP,1.167,1.0,_4,0.3684,0.9934
1,ResNet Block,1.500,1.0,out,0.3333,0.9956


## Category Filtering and Graph Neighborhood Queries

Filter nodes by quadrant category and query graph neighbors.

In [5]:
# Query Type I critical nodes
type1_nodes = engine.get_nodes("ResNet Block", category="Type I")
print("Type I Nodes in ResNet Block:")
for n in type1_nodes:
    print(f"  ID: {n['node_id']:<10} Type: {n['type']:<15} Grad RMS: {n['gradient_rms']:.4f}")

# Inspect neighbors of 'add'
neighbors = engine.get_neighbors("add", arch="ResNet Block")
print()
print("DAG Neighbors for 'add':")
print(f"  Predecessors (Inputs) : {neighbors['predecessors']}")
print(f"  Successors   (Outputs): {neighbors['successors']}")
print(f"  Siblings     (Co-in)  : {neighbors['siblings']}")

Type I Nodes in ResNet Block:
  ID: out        Type: call_module     Grad RMS: 1.0000
  ID: r2         Type: call_module     Grad RMS: 0.2409
  ID: x          Type: placeholder     Grad RMS: 0.1718

DAG Neighbors for 'add':
  Predecessors (Inputs) : ['l2', 'x']
  Successors   (Outputs): ['r2']
  Siblings     (Co-in)  : ['l1']
